# 🏴‍☠️ OpenViking en Google Colab

Este notebook permite correr el agente **OpenViking** directamente en Google Colab.

### 1. Configuración de Secretos
Ve al icono de la llave (🔑) a la izquierda y añade las siguientes claves:
- `TELEGRAM_TOKEN` (si vas a usar el bot)
- `GROQ_API_KEY` (Opcional, ahora usamos Ollama local)
- `OPENROUTER_API_KEY` (Opcional)

In [ ]:
# @title ⚙️ Configurar Entorno y Clonar
import os
repo_url = "https://github.com/codigo8a/OpenViking-Python.git"
branch = "google-colab"
repo_name = "OpenViking-Python"

if not os.path.exists('agent.py'):
    print("📥 Descargando OpenViking...")
    !git clone -b {branch} {repo_url}
    %cd {repo_name}
else:
    print("✅ Proyecto ya configurado.")

!pip install -q requests python-telegram-bot qdrant-client python-dotenv

In [ ]:
# @title 🚀 Levantar Ollama (Local LLM)
# Asegurar que zstd esté instalado (necesario para la extracción de Ollama)
!apt-get update && apt-get install -y zstd

!curl -fsSL https://ollama.com/install.sh | sh

import subprocess
import time
import os

# Iniciar Ollama en segundo plano
print('⏳ Iniciando Ollama...')
subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(15)  # Esperar a que inicie el servidor

# Descargar el modelo llama3 (8B) que cabe perfectamente en los 15GB de Colab
!ollama pull llama3:8b
print('✅ Ollama listo con Llama3:8b')

In [ ]:
# @title 🤖 Iniciar Agente (Modo CLI)
from agent import OpenVikingAgent

agent = OpenVikingAgent()
print("✅ OpenViking listo.")

task = "lista los archivos actuales" # @param {type:"string"}
response = agent.execute_task(task)
print("\n--- RESPUESTA ---")
print(response)

In [ ]:
# @title 🚀 Iniciar Bot de Telegram
from google.colab import userdata
import os

token = userdata.get('TELEGRAM_TOKEN')
if token:
    os.environ['TELEGRAM_TOKEN'] = token
    print("📡 Iniciando servidor de Telegram...")
    # Inyectamos el token directamente en la ejecución del script
    !TELEGRAM_TOKEN="{token}" python telegram_bot.py
else:
    print("❌ ERROR: No se encontró TELEGRAM_TOKEN en tus Secretos.")